# 05 - Gold Mart: NYC Taxi Trip Analytics

## Purpose
Build analytics-ready Gold tables from the Silver Green Taxi and Weather datasets.

## Models created
- `dim_datetime`
- `dim_location`
- `dim_weather`
- `fact_taxi_trips`

## Fact grain
One row represents one Green Taxi trip.

## Why this layer matters
This layer organizes cleaned data into a star schema so it is easier and faster to use for reporting, dashboards, and business analysis.


In [0]:
%sql

SELECT * 
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

SELECT * 
FROM `ftw-week-08`.`02_silver`.weather
LIMIT 10;

## 1. Inspect Silver Source Tables

Before building the Gold/Mart layer, inspect the available Green Taxi and Weather columns.

This confirms the exact field names, data types, and possible join keys that will be used for the dimensional model.

In [0]:
%sql
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

## 2. Inspect Green Taxi Schema

Review the Silver Green Taxi table to identify the exact trip, pickup/dropoff datetime, location, passenger, distance, fare, and payment columns.

These fields will be used to build the Gold dimensions and the `fact_taxi_trips` table.

In [0]:
%sql
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

## 3. Build `dim_datetime`

Create an hourly datetime dimension from Green Taxi pickup timestamps.

This standardizes time-based analysis, such as trips by date, day of week, month, and pickup hour. It will also provide the hourly key needed to join taxi trips with weather data.

In [0]:
%sql
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_datetime AS
SELECT DISTINCT
  date_trunc('HOUR', lpep_pickup_datetime) AS datetime_hour,
  CAST(lpep_pickup_datetime AS DATE) AS trip_date,
  YEAR(lpep_pickup_datetime) AS trip_year,
  MONTH(lpep_pickup_datetime) AS trip_month,
  DAY(lpep_pickup_datetime) AS trip_day,
  DAYOFWEEK(lpep_pickup_datetime) AS day_of_week_number,
  DATE_FORMAT(lpep_pickup_datetime, 'EEEE') AS day_of_week_name,
  HOUR(lpep_pickup_datetime) AS pickup_hour
FROM `ftw-week-08`.`02_silver`.green_taxi
WHERE lpep_pickup_datetime IS NOT NULL;

## 4. Validate `dim_datetime`

Check that the Gold datetime dimension was created successfully and contains unique hourly records.

The validation confirms the table is not empty and that each `datetime_hour` appears only once.

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT datetime_hour) AS unique_datetime_hours,
  MIN(datetime_hour) AS earliest_hour,
  MAX(datetime_hour) AS latest_hour
FROM `ftw-week-08`.`03_gold`.dim_datetime;

## 5. Build `dim_location`

Create a location dimension containing all unique pickup and drop-off location IDs.

Combining both location columns ensures that every location used in the taxi trips can be referenced consistently by the fact table.

In [0]:
%sql
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_location AS
SELECT DISTINCT
  location_id
FROM (
  SELECT PULocationID AS location_id
  FROM `ftw-week-08`.`02_silver`.green_taxi
  WHERE PULocationID IS NOT NULL

  UNION

  SELECT DOLocationID AS location_id
  FROM `ftw-week-08`.`02_silver`.green_taxi
  WHERE DOLocationID IS NOT NULL
);

In [0]:
%sql
--Validation--
SELECT COUNT(*) AS total_locations
FROM `ftw-week-08`.`03_gold`.dim_location;

## 6. Build `dim_weather`

Create a weather dimension at hourly grain using the Silver Weather table.

The hourly timestamp will serve as the join key between weather conditions and taxi trip pickup time.

In [0]:
%sql
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_weather AS
SELECT
  date_trunc('HOUR', weather_datetime) AS weather_hour,
  temperature_2m,
  precipitation,
  rain,
  snowfall,
  weather_code,
  wind_speed_10m,
  source_system,
  source_url,
  batch_id,
  ingested_at
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY date_trunc('HOUR', weather_datetime)
      ORDER BY ingested_at DESC
    ) AS row_num
  FROM `ftw-week-08`.`02_silver`.weather
  WHERE weather_datetime IS NOT NULL
)
WHERE row_num = 1;

## 7. Validate `dim_weather`

Validate that the weather dimension contains hourly records and that each `weather_hour` is unique.

This ensures the weather table can be safely joined to taxi trips without creating duplicate fact rows.

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT weather_hour) AS unique_weather_hours,
  MIN(weather_hour) AS earliest_hour,
  MAX(weather_hour) AS latest_hour
FROM `ftw-week-08`.`03_gold`.dim_weather;

## 8. Build `fact_taxi_trips`

Create the main Gold fact table at one-row-per-taxi-trip grain.

The fact table will contain trip identifiers, datetime keys, pickup and drop-off location keys, weather information, trip metrics, fare details, and pipeline lineage columns.

In [0]:
%sql
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.fact_taxi_trips AS

WITH taxi_source AS (
  SELECT
    t.*,
    ROW_NUMBER() OVER (
      ORDER BY
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        PULocationID,
        DOLocationID
    ) AS source_row_number
  FROM `ftw-week-08`.`02_silver`.green_taxi t
)

SELECT
  sha2(
    concat_ws(
      '||',
      CAST(t.VendorID AS STRING),
      CAST(t.lpep_pickup_datetime AS STRING),
      CAST(t.lpep_dropoff_datetime AS STRING),
      CAST(t.PULocationID AS STRING),
      CAST(t.DOLocationID AS STRING),
      CAST(t.source_row_number AS STRING)
    ),
    256
  ) AS trip_id,

  dt.datetime_hour,
  t.VendorID AS vendor_id,
  t.lpep_pickup_datetime,
  t.lpep_dropoff_datetime,

  t.PULocationID AS pickup_location_id,
  t.DOLocationID AS dropoff_location_id,

  t.passenger_count,
  t.trip_distance,
  t.fare_amount,
  t.extra,
  t.mta_tax,
  t.tip_amount,
  t.tolls_amount,
  t.improvement_surcharge,
  t.total_amount,
  t.payment_type,
  t.trip_type,

  w.temperature_2m,
  w.precipitation,
  w.rain,
  w.snowfall,
  w.weather_code,
  w.wind_speed_10m,

  t.batch_id,
  current_timestamp() AS mart_created_at

FROM taxi_source t

LEFT JOIN `ftw-week-08`.`03_gold`.dim_datetime dt
  ON date_trunc('HOUR', t.lpep_pickup_datetime) = dt.datetime_hour

LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather w
  ON date_trunc('HOUR', t.lpep_pickup_datetime) = w.weather_hour;

## 9. Validate `fact_taxi_trips`

Validate the fact table row count, trip-key uniqueness, and datetime coverage.

The checks confirm that the fact table follows the expected one-row-per-trip grain and that the Gold joins were completed successfully.

In [0]:
%sql
SELECT
  COUNT(*) AS total_trips,
  COUNT(DISTINCT trip_id) AS unique_trip_ids,
  COUNT(*) - COUNT(DISTINCT trip_id) AS duplicate_trip_ids,
  COUNT(datetime_hour) AS matched_datetime_rows,
  COUNT(temperature_2m) AS matched_weather_rows
FROM `ftw-week-08`.`03_gold`.fact_taxi_trips;

### Validation Result

The fact table contains 133,367 taxi trips with 133,367 unique trip IDs and zero duplicate records. All trips matched the datetime dimension. Weather data matched 133,356 trips; the remaining 11 trips had no corresponding hourly weather record and were retained through the LEFT JOIN.